In [1]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib

# add path to project root
import sys
PROJECT_ROOT = Path.cwd()

if "notebooks" in PROJECT_ROOT.parts:
    PROJECT_ROOT = PROJECT_ROOT.parent
    
sys.path.append(str(PROJECT_ROOT))
    
    
from src.preprocess.gaming_text_labeler import GamingTextLabeler
from src.ensemble.ensemble import Ensemble
from src.model.social_media_collection import SocialMediaModelCollection
from src.ensemble.score import ClassificationMetrics

# instantiate preprocessor and labeler
gaming_labeler = GamingTextLabeler()
metric = ClassificationMetrics()

In [2]:
CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'model_dir': PROJECT_ROOT / 'models',
    'text_col': 'message',
    'label_col': 'label'
}

np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

In [3]:
social_multi_paths = list((CONFIG['model_dir'] / 'multi' ).glob('social_media_*.joblib'))
scaler_multi_path = CONFIG['model_dir'] / 'helper' / 'multi_max_abs_scaler.joblib'
nb_tfidf_multi_path = CONFIG['model_dir'] / 'helper' / 'multi_word_tfidf_vectorizer.joblib'

In [4]:
# load WOT train data
d = CONFIG['data_dir']

wot_train = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train["data_source"] = "wot"

# load DOTA training data
dota_train = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train["data_source"] = "dota"


# combine datasets
train = pd.concat([wot_train, dota_train], ignore_index=True)

wot_val = pd.read_parquet(d / 'wot' / 'x_validation.parquet')
wot_val["data_source"] = "wot"
dota_val = pd.read_parquet(d / 'dota' / 'x_validation.parquet')
dota_val["data_source"] = "dota"

# combine holdout datasets
val = pd.concat([wot_val, dota_val], ignore_index=True)

In [5]:
# create 3-class
train = gaming_labeler.convert_three_class(
    train, 
    label_column=lc, 
    output_column='label_3class',
    data_source_column='data_source'
)
val = gaming_labeler.convert_three_class(
    val, 
    label_column=lc, 
    output_column='label_3class',
    data_source_column='data_source'
)

In [6]:
social_media_collection = SocialMediaModelCollection(
    model_joblibs=social_multi_paths,
    scaler_path=scaler_multi_path,
    nb_tfidf_path=nb_tfidf_multi_path,
)

ensemble_multi = Ensemble(model_collections=[social_media_collection])

In [7]:
social_media_collection.classifiers

{PosixPath('/Users/paolacalle/Desktop/NYU/semesters/spring-2026/ML/hw/gaming-toxicity-detection/models/multi/social_media_multioutput(lr-cv)_pipeline.joblib'): Pipeline(steps=[('clf',
                  MultiOutputClassifier(estimator=LogisticRegressionCV(class_weight='balanced',
                                                                       cv=3,
                                                                       max_iter=1000,
                                                                       random_state=42,
                                                                       scoring='f1_macro',
                                                                       solver='liblinear'),
                                        n_jobs=2))]),
 PosixPath('/Users/paolacalle/Desktop/NYU/semesters/spring-2026/ML/hw/gaming-toxicity-detection/models/multi/social_media_multioutput(nb)_pipeline.joblib'): Pipeline(steps=[('clf',
                  MultiOutputClassifier(estimator=C

In [8]:
# simple majority vote ensemble
train_pred = ensemble_multi.predict_simple_majority(train[tc])
metric.metrics(train['label_3class'], train_pred)

Predicting with SimpleMajority...
Input has 53707 samples.
social_media_multioutput(lr-cv)_pipeline: (53707,)
social_media_multioutput(nb)_pipeline: (53707,)
social_media_multioutput(svc)_pipeline: (53707,)
Collected predictions from 3 models.
Prediction matrix shape: (53707, 3)
Aggregating predictions from 3 models...


{'CV Macro F1': 0.1422975366785835,
 'CV Weighted F1': 0.6774301797773907,
 'Accuracy': 0.6738972573407563,
 'Coverage': np.float64(1.0),
 'Precision': 0.15680191644647884}

In [9]:
val_pred = ensemble_multi.predict_simple_majority(val[tc])
metric.metrics(val['label_3class'], val_pred)

Predicting with SimpleMajority...
Input has 17056 samples.
social_media_multioutput(lr-cv)_pipeline: (17056,)
social_media_multioutput(nb)_pipeline: (17056,)
social_media_multioutput(svc)_pipeline: (17056,)
Collected predictions from 3 models.
Prediction matrix shape: (17056, 3)
Aggregating predictions from 3 models...


{'CV Macro F1': 0.13122658603474785,
 'CV Weighted F1': 0.6597838567985445,
 'Accuracy': 0.6387195121951219,
 'Coverage': np.float64(1.0),
 'Precision': 0.14686504046340704}

In [10]:
res = ensemble_multi.fit_weighted_majority(
    train[tc], 
    train['label_3class'],
    metric.score
)

In [11]:
# simple majority vote ensemble
train_pred = ensemble_multi.predict_weighted_majority(train[tc], weights=res[0])
metric.metrics(train['label_3class'], train_pred)

{'CV Macro F1': 0.14265686836502964,
 'CV Weighted F1': 0.6770950127177259,
 'Accuracy': 0.6728359431731432,
 'Coverage': np.float64(1.0),
 'Precision': 0.158045976579648}

In [12]:
# simple majority vote ensemble
val_pred = ensemble_multi.predict_weighted_majority(val[tc], weights=res[0])
metric.metrics(val['label_3class'], val_pred)

{'CV Macro F1': 0.13124218058534753,
 'CV Weighted F1': 0.6588036683993089,
 'Accuracy': 0.6369019699812383,
 'Coverage': np.float64(1.0),
 'Precision': 0.14106022239007418}

In [13]:
res = ensemble_multi.fit_weighted_confidence_majority(
    train[tc], 
    train['label_3class'],
    metric.score
)

Constructing confidence tensor...


IndexError: shape mismatch: indexing arrays could not be broadcast together with shapes (53707,) (53707,6) 

In [ ]:
train_pred = ensemble_multi.predict_weighted_confidence_majority(train[tc], weights=res[0], threshold=res[1])
metric.metrics(train['label_3class'], train_pred)

Constructing confidence tensor...
Total models in ensemble: 3
Expected confidence shape: (n_samples=53707, n_classes=3)
Predicted labels shape: (53707,)


{'CV Macro F1': 0.4926608985359388,
 'CV Weighted F1': 0.7334063355142145,
 'Accuracy': 0.7350252294859143,
 'Coverage': np.float64(1.0),
 'Precision': 0.5075013740343106}

In [ ]:
train_pred = ensemble_multi.predict_weighted_confidence_majority(val[tc], weights=res[0], threshold=res[1])
metric.metrics(val['label_3class'], train_pred)

Constructing confidence tensor...
Total models in ensemble: 3
Expected confidence shape: (n_samples=17056, n_classes=3)
Predicted labels shape: (17056,)


{'CV Macro F1': 0.4405022577057946,
 'CV Weighted F1': 0.7012784215383212,
 'Accuracy': 0.6870895872420263,
 'Coverage': np.float64(1.0),
 'Precision': 0.4584686695914271}

In [ ]:
res